In [11]:
%pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import faiss
import numpy as np
import ollama

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [13]:
pdf_path = "academic_document.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print("Document loaded successfully!")
print("Characters:", len(text))

Document loaded successfully!
Characters: 11


In [14]:
def create_chunks(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

chunks = create_chunks(text)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

Number of chunks: 1

First chunk:















In [15]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (1, 384)


In [16]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings.astype("float32"))

print("Embeddings stored successfully!")
print("Total vectors:", index.ntotal)

Embeddings stored successfully!
Total vectors: 1


In [17]:
question = input("Enter your question: ")

print("\nUser Question:")
print(question)

Enter your question:  "what is logistic regression"



User Question:
"what is logistic regression"


In [18]:
question_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True
)

k = 3

distances, indices = index.search(
    question_embedding.astype("float32"),
    k
)

retrieved_context = ""

for i in indices[0]:
    retrieved_context += chunks[i] + "\n\n"

print("\nRetrieved Relevant Context:\n")
print(retrieved_context)


Retrieved Relevant Context:











































In [19]:
prompt = f"""
You are an academic assistant.

Answer the user's question using only the information
provided in the retrieved context.

If the answer is not available in the context,
say "The answer is not available in the document."

Retrieved Context:
{retrieved_context}

User Question:
{question}

Answer:
"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response["message"]["content"]

print("\nFinal Response:\n")
print(answer)


Final Response:

Logistic regression is a supervised learning algorithm used for classification problems. It's a statistical technique that models the probability of an event occurring based on one or more independent variables. The output of the model is a probability value between 0 and 1, where 0 indicates the event is unlikely to occur, and 1 indicates it's likely to occur.
